# 08. Person Re-ID - casvit_xs

| 항목 | 내용 |
|---|---|
| 대상 모델 | `casvit_xs.dxnn` |
| 카테고리 | **Person Re-ID** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 분류용 출력 자리를 특징 벡터로 쓰는 구성과 임베딩 전용 학습의 차이를 유사도 값으로 비교해 설명할 수 있다.
2. 같은 사람 유사도가 임계값에 겨우 걸리는 상황에서 임계값을 정하는 판단을 근거와 함께 제시할 수 있다.
3. NPU 가 낸 여섯 개 중간 값이 CPU Task 에서 하나로 합쳐지는 흐름을 Task 구성으로 설명할 수 있다.
4. 전신 사진을 정사각으로 눌러 넣는 전처리가 이 작업에서 갖는 위험을 설명할 수 있다.

## 1. Person Re-ID (사람 재식별)

**사람이 찍힌 두 장이 같은 사람인지 판정하기 위해 사진을 숫자 목록으로 바꾸는 작업**

얼굴이 보이지 않는 각도나 먼 거리에서도 옷·체형·자세·색 조합 같은 단서로 같은 사람을 이어 붙인다. 카메라가
여러 대인 환경에서 사람의 이동을 잇는 데 쓰인다.

**출력의 성격**

이름표가 아니라 좌표이며, 한 장만으로는 뜻이 없고 둘을 비교해야 뜻이 생긴다. 옷차림에 크게 의존하므로 같은
사람이 옷을 갈아입으면 다른 사람으로, 다른 사람이 같은 옷을 입으면 같은 사람으로 판정될 수 있다.

**활용 사례**

- 여러 카메라에 걸친 동일 인물 추적
- 매장 안 동선 분석
- 놓친 대상을 다른 화면에서 다시 찾기

**포함되지 않는 작업**

- 사진에서 사람 찾기 (Object Detection)
- 신원 확인 (Face Embedding)
- 사람 수 세기 (Object Detection)

## 2. 전체 파이프라인

```text
sample/img/person_pair/  정렬된 파일 3장을 차례로
  → 전처리         BGR→RGB · 224×224 직접 축소 · uint8 유지
  → run()          한 장마다 입력 Tensor 하나를 전달
  → npu_0          중간 tensor 6개 (내부 전달, Public 아님)
  → cpu_0          output  [1, 1000]  float32  (Public Output)
  → 후처리         flatten → L2 정규화 → EmbeddingResult
  → 비교           첫 장을 기준으로 두고 나머지와 코사인 유사도 계산
  → 시각화         기준과 현재를 좌우로 붙이고 유사도·판정을 아래 띠에 적음
```

![08. Person Re-ID 전체 파이프라인](resources/08_Person_Re-ID_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `input` |
| Shape / Dtype | `[1, 224, 224, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | 직접 축소 (비율 유지 없음) |
| Normalization | 없음 |

**정사각 입력이 이 작업에서 갖는 위험**

- **사람은 세로로 길다** — 세로로 긴 보행자 사진을 224×224 로 누르면 가로로 퍼진 형태가 되고, 체형이 단서인
  작업에서 그 눌림이 사람마다 다르게 작용한다. 이번 sample 은 1024×1024 정사각이라 눌림이 없으므로 이 위험은
  드러나지 않는다
- **사람이 이미 잘려 있어야 한다** — 이 모델에는 사람을 찾는 단계가 없다. 배경이 많이 섞이면 배경이 특징에
  들어가 같은 장소의 다른 사람이 비슷해진다
- **사진 입력만 받는다** — 영상·카메라 입력은 실행 진입점에서 거부된다

### 3.2 출력

이 모델은 `.dxnn` 하나 안에 Task 가 둘이다. `npu_0` 이 중간 값 여섯 개를 내고 `cpu_0` 이 그것을 받아 최종 벡터
하나를 만든다.

| 생성 Task | Tensor | shape / dtype | 다음 사용자 | Public |
|---|---|---|---|---|
| `npu_0` | `add` · `split_split_0` · `split_split_1` · `..._mul_out` | `[1, 48, 56, 56]` / `float32` | `cpu_0` | 아님 — 내부 전달 |
| `npu_0` | `sigmoid` · `sigmoid_2` | `[1, 1, 56, 56]` / `float32` | `cpu_0` | 아님 — 내부 전달 |
| `cpu_0` | `output` | `[1, 1000]` / `float32` | Host 후처리 | CPU Task 출력 (= Public 출력) |

`Public` 은 `InferenceEngine.run()` 이 돌려주는 값인지를 가리키는 표시다. 중간 값 여섯 개는 합쳐서 2.3 MB 가
넘지만 밖으로 나오지 않고, 4,000 바이트짜리 `output` 하나만 손에 들어온다.

**shape 각 자리의 뜻**

| index | 값 | 뜻 |
|---|---:|---|
| 0 | 1 | 한 번에 처리하는 사진 장수 |
| 1 | 1000 | 특징 벡터의 차원 수 |

**1,000 이라는 차원 수가 이 모델의 성격을 알려 준다.** 임베딩 전용 모델은 128·512 처럼 목적에 맞춘 차원을 쓰지만,
이 값은 ImageNet 부류 수와 같다. **분류 모델의 마지막 출력 자리를 그대로 특징 벡터로 쓰고 있다**는 뜻이다.

그래서 값의 성격도 다르다. 이번 실행의 원시 값은 `-5.30`~`9.70` 으로 폭이 크며, 부류 점수(logit)의 범위다.
같은 방향인지를 재는 데는 쓸 수 있지만, **같은 사람끼리 가깝게 모이도록 따로 학습된 좌표가 아니다.**

**그 차이가 유사도 값에 그대로 나타난다.** 얼굴 임베딩 전용 모델은 같은 얼굴에서 0.9 를 넘기지만, 이 모델은
같은 사람의 다른 사진에서 `0.4527` 로 임계값 `0.4` 를 겨우 넘었다. 다른 사람은 `0.1961` 이다. 두 값의 간격이
좁으므로 **임계값을 조금만 올려도 같은 사람을 놓친다.**

비교는 방향으로 하며 계산은 코사인 유사도다.

```text
cos(a, b) = (a · b) / (|a| × |b|)
```

후처리가 길이를 1 로 맞추므로 분모가 `1 × 1` 이 되어 이후 비교는 내적 한 번으로 끝난다. 이번 실행에서 기준
사진의 원시 벡터 길이는 `49.1222` 였다.

### 3.3 Config.json

| key | 현재 값 | 기본값 | 의미 |
|---|---:|---:|---|
| `top_k` | 5 | 5 | 이 모델의 후처리가 읽지 않는 key |

**이 파일의 `top_k` 는 쓰이지 않는다.** 임베딩 후처리는 `normalize` 하나만 설정에서 찾고, 그 key 는 파일에
없으므로 코드 기본값 `True` 가 적용된다. 상위 몇 개를 남기는 일은 분류 계열 후처리의 것이며, 예제 폴더를 같은
형태로 맞추다 남은 것으로 판단된다.

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. `npu_0` 이 중간 값
여섯 개를 내고 `cpu_0` 만 model output 을 갖는지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "casvit_xs.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 디렉토리를 정하고 설정 파일을 읽는다. 이후 셀은 여기서 만든 이름을 그대로 쓴다.

In [ ]:
import json
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

sys.dont_write_bytecode = True
IMAGE_DIR = DX_APP / "sample" / "img/person_pair"
CONFIG_PATH = DX_APP / "src" / "python_example" / "reid" / "casvit_xs" / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}

IMAGE_PATHS = sorted(IMAGE_DIR.glob("*.jpg"))
THRESHOLD = 0.4

print("config", CONFIG)
print("images", [path.name for path in IMAGE_PATHS])

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 이 모델의 후처리는 되돌릴
  좌표가 없어 받기만 하고 읽지 않는다
- `EmbeddingResult` — 후처리가 만들어 비교 단계로 넘기는 결과. 정규화된 벡터와 모델 종류를 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale_x: float = 0.0
    scale_y: float = 0.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class EmbeddingResult:
    """정규화된 특징 벡터. 원본 대응: common/base/i_processor.py"""

    embedding: np.ndarray = field(default=None)
    model_type: str = ""


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. 중간 값 여섯 개는 목록에 없고 `output`
하나만 보이는지 확인한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output:", output_info)

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
EMBED_DIM = output_info[0]["shape"][1]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT} / feature dim = {EMBED_DIM}")

### 4.2 전처리

사람 사진 세 장을 각각 RGB 224×224 `uint8` 로 바꾼다. 세 장 모두 같은 절차를 거쳐야 벡터끼리 비교할 수 있다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 224×224 정사각
- dtype — `uint8` 유지
- 축소 비율 — 원본 비율에 따라 축마다 다름

#### 입력 이미지 확인

세 장을 읽어 나란히 표시한다. 파일명이 순서를 정하고 첫 장이 기준이 된다.

In [ ]:
images = [cv2.imread(str(path)) for path in IMAGE_PATHS]
for path, img in zip(IMAGE_PATHS, images):
    print(f"{path.name:18s} shape={img.shape} dtype={img.dtype}")

preview_height = 280
previews = [cv2.resize(img, (int(img.shape[1] * preview_height / img.shape[0]), preview_height))
            for img in images]
notebook_show_bgr(cv2.hconcat(previews), "1_reference | 2_same | 3_different", figsize=(11, 5))

#### 전처리 함수 정의

- `casvit_preprocessor_create` — 모델 입력 크기를 state 에 담음
- `casvit_preprocessor_process` — 사진 한 장을 입력 Tensor 와 Context 로 변환

In [ ]:
def casvit_preprocessor_create(input_width: int, input_height: int) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height}


def casvit_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]
    ctx.scale_x = float(ctx.input_width) / float(ctx.original_width)
    ctx.scale_y = float(ctx.input_height) / float(ctx.original_height)

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (ctx.input_width, ctx.input_height),
                         interpolation=cv2.INTER_LINEAR)
    return resized, ctx

#### 전처리 실행

세 장을 같은 함수로 처리해 입력 Tensor 목록을 만든다. 세 번째 사진의 축소 비율이 앞의 둘과 다른 것은 원본
크기가 달라서다.

In [ ]:
preprocessor_state = casvit_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
tensors, contexts = [], []
for path, img in zip(IMAGE_PATHS, images):
    tensor, ctx = casvit_preprocessor_process(preprocessor_state, img)
    tensors.append(tensor)
    contexts.append(ctx)
    print(f"{path.name:18s} tensor={tensor.shape} {tensor.dtype} "
          f"scale_x={ctx.scale_x:.4f} scale_y={ctx.scale_y:.4f}")

### 4.3 추론

#### DXNN Inference 실행

세 장을 차례로 넘긴다. 한 번의 `run()` 은 한 장만 처리하므로 비교를 하려면 여러 번 호출해야 한다. 출력되는
시간은 각 1회 측정값이고 런타임과의 왕복 비용이 포함돼 있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
raw_outputs = []
for path, tensor in zip(IMAGE_PATHS, tensors):
    started = time.perf_counter()
    outputs = engine.run([tensor])
    elapsed_ms = (time.perf_counter() - started) * 1000.0
    array = np.asarray(outputs[0])
    raw_outputs.append(outputs)
    print(f"{path.name:18s} {elapsed_ms:6.2f} ms  output {len(outputs)}개  shape={array.shape} "
          f"min={array.min():7.4f} max={array.max():7.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- 출력 개수 — 중간 값 여섯 개는 나오지 않고 하나만 온다
- 값의 폭 — 부류 점수 범위라 임베딩 전용 모델보다 훨씬 넓다
- 길이 — 사진마다 다르므로 정규화가 필요하다

In [ ]:
for path, outputs in zip(IMAGE_PATHS, raw_outputs):
    raw = np.asarray(outputs[0]).flatten()
    print(f"{path.name:18s} L2={np.linalg.norm(raw):8.4f} "
          f"min={raw.min():8.4f} max={raw.max():8.4f} 음수={int((raw < 0).sum())}/{raw.size}")

print("\nreference 앞 5개 값:", np.array2string(
    np.asarray(raw_outputs[0][0]).flatten()[:5], precision=4))

### 4.4 후처리

벡터를 1차원으로 펴고 길이 1 로 맞춘다. 순서는 flatten → L2 정규화이며, 이 모델은 07번 얼굴 임베딩과 같은
후처리를 쓴다. **후처리가 같다는 사실이 곧 결과의 성격이 같다는 뜻은 아니다** — 앞에 붙은 모델이 무엇을
학습했는지가 유사도의 폭을 정한다.

#### 후처리 함수 정의

- `casvit_postprocessor_create` — 정규화 여부를 state 에 담음
- `casvit_postprocessor_process` — flatten 과 L2 정규화를 거쳐 Result 생성
- `casvit_cosine_similarity` — 두 벡터의 코사인 유사도

In [ ]:
def casvit_postprocessor_create(config: dict) -> dict:
    """Config 와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {"normalize": config.get("normalize", True)}


def casvit_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    embedding = np.asarray(outputs[0]).flatten().astype(np.float32)
    if state["normalize"]:
        norm = np.linalg.norm(embedding)
        if norm > 0:
            embedding = embedding / norm
    return [EmbeddingResult(embedding=embedding, model_type="face_embedding")]


def casvit_cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """두 벡터가 같은 방향을 보는 정도. 원본 대응: EmbeddingVisualizer._cosine_similarity()"""
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
    if norm_a < 1e-8 or norm_b < 1e-8:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))

#### 후처리 실행

확인할 것:

- 정규화 후 길이 — 셋 다 `1.000000` 인지
- 같은 사람과 다른 사람의 유사도 — 임계값 `0.4` 를 사이에 두고 갈라지는지
- 두 값의 간격 — 임계값을 어디에 두어야 안전한지 판단할 근거

`model_type` 이 `face_embedding` 으로 나오는 것은 이 모델이 얼굴 임베딩용 후처리를 그대로 쓰기 때문이며,
사람 전신을 다루는 이 작업에서는 이름이 실제 쓰임과 어긋난다.

In [ ]:
postprocessor_state = casvit_postprocessor_create(CONFIG)
embeddings = []
for path, outputs in zip(IMAGE_PATHS, raw_outputs):
    result = casvit_postprocessor_process(postprocessor_state, outputs, contexts[0])[0]
    embeddings.append(result.embedding)
    print(f"{path.name:18s} dim={result.embedding.size} "
          f"L2={np.linalg.norm(result.embedding):.6f} model_type={result.model_type!r}")

print("\n기준(1_reference) 과의 유사도:")
similarities = []
for path, embedding in zip(IMAGE_PATHS[1:], embeddings[1:]):
    similarity = casvit_cosine_similarity(embeddings[0], embedding)
    similarities.append(similarity)
    verdict = "SAME" if similarity > THRESHOLD else "DIFFERENT"
    print(f"  {path.name:18s} {similarity:.4f} → {verdict} (임계값 {THRESHOLD})")

print(f"\n두 값의 간격 = {similarities[0] - similarities[1]:.4f}")
print(f"임계값을 {similarities[0]:.4f} 보다 높이면 같은 사람도 DIFFERENT 로 판정된다.")

### 4.5 시각화

#### Visualizer 코드

기준 사진을 안에 담아 두고, 다음 사진이 올 때마다 좌우로 붙인 그림과 유사도·판정을 만든다. 첫 호출은 기준을
담기만 하고 그림을 내지 않으므로 세 장을 넣으면 그림은 두 장 나온다. 그림에 넣는 글자는 영문만 쓴다 —
OpenCV 기본 폰트에 한글 자모가 없어 한글을 넣으면 글자가 깨진다.

In [ ]:
class CasvitVisualizer:
    """기준 사진을 담아 두고 다음 사진과 좌우 비교 그림을 만든다. 원본 대응: Runner.visualize()"""

    def __init__(self, threshold: float = 0.4):
        self.threshold = threshold
        self._ref_image = None
        self._ref_embedding = None

    def visualize(self, image: np.ndarray, results: list):
        embedding = np.asarray(results[0].embedding, dtype=np.float32)
        if self._ref_image is None:
            self._ref_image = image.copy()
            self._ref_embedding = embedding
            return None

        similarity = casvit_cosine_similarity(self._ref_embedding, embedding)
        return self._canvas(self._ref_image, image, similarity)

    @staticmethod
    def _fit(image: np.ndarray, box_w: int, box_h: int) -> np.ndarray:
        height, width = image.shape[:2]
        scale = min(box_w / width, box_h / height)
        return cv2.resize(image, (int(width * scale), int(height * scale)))

    def _canvas(self, ref: np.ndarray, cur: np.ndarray, similarity: float) -> np.ndarray:
        canvas_h, canvas_w, bar_h, gap = max(cur.shape[0], 640), max(cur.shape[1], 960), 60, 6
        image_h, half_w = canvas_h - bar_h, (canvas_w - gap) // 2
        canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

        for index, source in enumerate((ref, cur)):
            fitted = self._fit(source, half_w, image_h)
            height, width = fitted.shape[:2]
            x = index * (half_w + gap) + (half_w - width) // 2
            y = (image_h - height) // 2
            canvas[y:y + height, x:x + width] = fitted
            cv2.putText(canvas, ("Reference", "Current")[index], (x + 4, y + 22),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2, cv2.LINE_AA)

        cv2.line(canvas, (half_w + gap // 2, 0), (half_w + gap // 2, image_h), (80, 80, 80), 1)

        overlay = canvas.copy()
        cv2.rectangle(overlay, (0, image_h), (canvas_w, canvas_h), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.7, canvas, 0.3, 0, canvas)

        matched = similarity > self.threshold
        color = (0, 200, 0) if matched else (0, 0, 220)
        text = f"Cosine Similarity: {similarity:.4f}  [{'SAME' if matched else 'DIFFERENT'}]"
        cv2.putText(canvas, text, (10, image_h + 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2,
                    cv2.LINE_AA)
        return canvas

#### 결과 시각화

세 장을 순서대로 넣어 비교 그림 두 장을 만든다. 첫 장은 기준으로만 쓰이므로 그림이 나오지 않는다.

In [ ]:
visualizer = CasvitVisualizer(threshold=THRESHOLD)
for path, img, embedding in zip(IMAGE_PATHS, images, embeddings):
    canvas = visualizer.visualize(img, [EmbeddingResult(embedding=embedding)])
    if canvas is None:
        print(f"{path.name:18s} 기준으로 저장 (그림 없음)")
        continue
    notebook_show_bgr(canvas, f"Reference vs {path.name}", figsize=(11, 7))

#### 임계값을 바꾸면 판정이 어떻게 갈리는지

같은 사람과 다른 사람의 유사도 간격이 좁으므로 임계값 선택이 결과를 바로 바꾼다. 값을 여러 개 넣어 두 판정이
어떻게 뒤집히는지 표로 확인한다.

In [ ]:
print(f"{'임계값':>8}  {'2_same':>10}  {'3_different':>12}")
for candidate in (0.15, 0.20, 0.30, 0.40, 0.45, 0.50):
    verdicts = ["SAME" if similarity > candidate else "DIFFERENT" for similarity in similarities]
    print(f"{candidate:8.2f}  {verdicts[0]:>10}  {verdicts[1]:>12}")
print(f"\n같은 사람 {similarities[0]:.4f} / 다른 사람 {similarities[1]:.4f} 사이에서만 "
      f"두 판정이 모두 맞는다.")